# Norse SNN — N-MNIST Event Camera Dataset

Trains a Spiking Neural Network using Norse on N-MNIST (digits recorded with a DVS event camera).

Runs **three experiments** back to back:
- `LIF` — Basic Leaky Integrate-and-Fire
- `LIFAdEx` — Adaptive Exponential LIF (threshold adapts after firing)
- `LIFRecurrent` — LIF with recurrent within-layer connections

All other parameters are kept identical so results are comparable.

**Runtime:** Enable GPU in Colab → `Runtime > Change runtime type > T4 GPU`

## 1. Install Dependencies

In [ ]:
# ── Install dependencies ───────────────────────────────────────
!pip install norse --quiet
!pip install tonic --no-deps --quiet
!pip install h5py importrosbag pbr --quiet

import numpy, torch
print(f"numpy : {numpy.__version__}   (should be 2.x)")
print(f"torch : {torch.__version__}")

In [ ]:
# ── Confirm all packages loaded correctly ─────────────────────
import numpy, torch, tonic, norse
print(f"numpy  : {numpy.__version__}   (should be 2.x)")
print(f"torch  : {torch.__version__}")
print(f"tonic  : {tonic.__version__}")
print(f"norse  : {norse.__version__}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT available — go to Runtime > Change runtime type > T4 GPU'}")

## 2. Imports

In [ ]:
import os
import time
import csv
import numpy as np
import torch
import torch.nn as nn
import tonic
import tonic.transforms as tonic_transforms
from torch.utils.data import DataLoader
from norse.torch import (
    LIFCell, LIFParameters,
    LIFAdExCell, LIFAdExParameters,
    LIFRecurrentCell,
)
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

## 3. Config
Change parameters here. `neuron_type` is set per-experiment in the experiment cells below.

In [ ]:
CONFIG = {
    # neuron_type is overridden per experiment — do not change here
    'neuron_type': 'LIF',

    # Neuron parameters
    'tau_mem'  : 0.02,   # membrane decay constant
    'threshold': 1.0,    # voltage to fire
    'surrogate': 'super',# surrogate gradient: 'super', 'heaviside', 'boxcar'

    # Architecture
    'num_layers' : 1,
    'hidden_size': 256,
    'output_size': 10,

    # Decoding: 'rate', 'max', 'first_spike'
    'decoding': 'rate',

    # Temporal
    'timesteps': 16,

    # Regularization
    'dropout_p'   : 0.0,
    'weight_decay': 1e-4,

    # Training
    'epochs'    : 5,
    'batch_size': 64,
    'lr'        : 1e-3,
    'optimizer' : 'adam',

    # Paths
    'data_dir'   : './data',
    'results_dir': './results',
}

## 4. Dataset — N-MNIST
Downloads N-MNIST (~1GB) on first run. Uses disk cache so subsequent epochs load instantly.

In [ ]:
def get_dataloaders(config):
    T = config['timesteps']
    frame_transform = tonic_transforms.ToFrame(
        sensor_size=tonic.datasets.NMNIST.sensor_size,
        n_time_bins=T,
    )
    train_raw = tonic.datasets.NMNIST(save_to=config['data_dir'], train=True,  transform=frame_transform)
    test_raw  = tonic.datasets.NMNIST(save_to=config['data_dir'], train=False, transform=frame_transform)

    train_dataset = tonic.DiskCachedDataset(train_raw, cache_path=f"{config['data_dir']}/cache_nmnist_T{T}_train")
    test_dataset  = tonic.DiskCachedDataset(test_raw,  cache_path=f"{config['data_dir']}/cache_nmnist_T{T}_test")

    def collate(batch):
        samples, labels = zip(*batch)
        x = torch.stack([torch.tensor(s, dtype=torch.float32) for s in samples])
        y = torch.tensor(labels, dtype=torch.long)
        return x, y

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True,  collate_fn=collate)
    test_loader  = DataLoader(test_dataset,  batch_size=config['batch_size'], shuffle=False, collate_fn=collate)

    print(f'Dataset      : N-MNIST (event camera)')
    print(f'Train samples: {len(train_dataset)}')
    print(f'Test samples : {len(test_dataset)}')
    print(f'Input shape  : (T={T}, 2 polarities, 34x34 pixels)')
    return train_loader, test_loader

train_loader, test_loader = get_dataloaders(CONFIG)
x, y = next(iter(train_loader))
print(f'Batch shape  : {x.shape}')

## 5. Model — Parameterized SNN

In [ ]:
INPUT_SIZE = 2 * 34 * 34  # 2312

def _lif_params(config):
    return LIFParameters(
        tau_mem_inv=torch.as_tensor(1.0 / config['tau_mem']),
        v_th=torch.as_tensor(config['threshold']),
        method=config['surrogate'],
    )

def _lifadex_params(config):
    return LIFAdExParameters(
        tau_mem_inv=torch.as_tensor(1.0 / config['tau_mem']),
        v_th=torch.as_tensor(config['threshold']),
        method=config['surrogate'],
    )

class SNN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config      = config
        self.neuron_type = config['neuron_type']
        self.decoding    = config['decoding']
        hidden           = config['hidden_size']
        out              = config['output_size']
        n_layers         = config['num_layers']
        dropout_p        = config['dropout_p']

        self.fc_layers = nn.ModuleList()
        self.lif_cells = nn.ModuleList()
        self.dropouts  = nn.ModuleList()
        in_size = INPUT_SIZE

        for _ in range(n_layers):
            if self.neuron_type == 'LIF':
                self.fc_layers.append(nn.Linear(in_size, hidden))
                self.lif_cells.append(LIFCell(p=_lif_params(config)))
            elif self.neuron_type == 'LIFAdEx':
                self.fc_layers.append(nn.Linear(in_size, hidden))
                self.lif_cells.append(LIFAdExCell(p=_lifadex_params(config)))
            elif self.neuron_type == 'LIFRecurrent':
                self.lif_cells.append(LIFRecurrentCell(
                    input_size=in_size, hidden_size=hidden, p=_lif_params(config)
                ))
            self.dropouts.append(nn.Dropout(p=dropout_p) if dropout_p > 0 else nn.Identity())
            in_size = hidden

        self.fc_out  = nn.Linear(hidden, out)
        self.lif_out = LIFCell(p=_lif_params(config))

    def forward(self, x):
        x = x.permute(1, 0, 2, 3, 4)
        T, B = x.shape[0], x.shape[1]
        x = x.reshape(T, B, -1)
        hidden_states = [None] * len(self.lif_cells)
        out_state = None
        output_spikes = []
        for t in range(T):
            z = x[t]
            for i, cell in enumerate(self.lif_cells):
                if self.neuron_type in ('LIF', 'LIFAdEx'):
                    z = self.fc_layers[i](z)
                    z, hidden_states[i] = cell(z, hidden_states[i])
                else:
                    z, hidden_states[i] = cell(z, hidden_states[i])
                z = self.dropouts[i](z)
            z = self.fc_out(z)
            z, out_state = self.lif_out(z, out_state)
            output_spikes.append(z)
        return self._decode(torch.stack(output_spikes, dim=0))

    def _decode(self, spikes):
        if self.decoding == 'rate':
            return spikes.sum(dim=0)
        elif self.decoding == 'max':
            return spikes.max(dim=0).values
        elif self.decoding == 'first_spike':
            T = spikes.shape[0]
            fired = spikes > 0
            first_t = torch.where(
                fired.any(dim=0),
                fired.float().argmax(dim=0).float(),
                torch.full_like(fired.float().argmax(dim=0).float(), float(T))
            )
            return (T - first_t).float()

def build_model(config):
    model = SNN(config)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model        : SNN ({config["neuron_type"]})')
    print(f'Hidden layers: {config["num_layers"]} x {config["hidden_size"]} neurons')
    print(f'Parameters   : {n:,}')
    return model

## 6. Training Loop

In [ ]:
def get_optimizer(model, config):
    opt = config['optimizer'].lower()
    if opt == 'adam':  return torch.optim.Adam(model.parameters(),  lr=config['lr'], weight_decay=config['weight_decay'])
    if opt == 'adamw': return torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    if opt == 'sgd':   return torch.optim.SGD(model.parameters(),   lr=config['lr'], weight_decay=config['weight_decay'], momentum=0.9)

def compute_spike_rate(model, x):
    model.eval()
    x_p  = x.permute(1, 0, 2, 3, 4)
    T, B = x_p.shape[0], x_p.shape[1]
    xf   = x_p.reshape(T, B, -1)
    states = [None] * len(model.lif_cells)
    spikes = []
    with torch.no_grad():
        for t in range(T):
            z = xf[t]
            for i, cell in enumerate(model.lif_cells):
                if model.neuron_type in ('LIF', 'LIFAdEx'):
                    z = model.fc_layers[i](z)
                    z, states[i] = cell(z, states[i])
                else:
                    z, states[i] = cell(z, states[i])
            spikes.append(z)
    return torch.stack(spikes, dim=0).mean().item()

def train(model, train_loader, config, results_dir):
    optimizer = get_optimizer(model, config)
    criterion = nn.CrossEntropyLoss()
    os.makedirs(results_dir, exist_ok=True)

    log_rows   = []
    all_losses = []
    all_accs   = []
    all_spkr   = []

    print(f'\nTraining {config["neuron_type"]} for {config["epochs"]} epochs on {DEVICE}\n')

    for epoch in range(1, config['epochs'] + 1):
        model.train()
        total_loss, total_correct, total_samples = 0.0, 0, 0
        spike_rates = []
        t0 = time.time()

        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            output = model(images)
            loss   = criterion(output, labels)
            loss.backward()
            optimizer.step()

            total_correct += (output.argmax(1) == labels).sum().item()
            total_samples += labels.size(0)
            total_loss    += loss.item()

            if batch_idx % 50 == 0:
                spike_rates.append(compute_spike_rate(model, images))
            if (batch_idx + 1) % 100 == 0:
                print(f'  Epoch {epoch}/{config["epochs"]} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}')

        epoch_time = round(time.time() - t0, 1)
        avg_loss   = round(total_loss / len(train_loader), 4)
        acc        = round(100.0 * total_correct / total_samples, 2)
        spkr       = round(100.0 * sum(spike_rates) / len(spike_rates), 2)

        all_losses.append(avg_loss)
        all_accs.append(acc)
        all_spkr.append(spkr)
        log_rows.append([epoch, avg_loss, acc, spkr, epoch_time])

        print(f'\nEpoch {epoch} | Loss: {avg_loss} | Acc: {acc}% | Spike rate: {spkr}% | Time: {epoch_time}s\n')

    # ── Training curves plot ──────────────────────────────────
    epochs_x = list(range(1, config['epochs'] + 1))
    fig, axes = plt.subplots(3, 1, figsize=(9, 10))
    fig.suptitle(f"Training — {config['neuron_type']}", fontsize=14)

    axes[0].plot(epochs_x, all_losses, marker='o', color='crimson')
    axes[0].set_title('Loss per Epoch')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].grid(True)

    axes[1].plot(epochs_x, all_accs, marker='o', color='steelblue')
    axes[1].set_title('Training Accuracy per Epoch')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].grid(True)

    axes[2].plot(epochs_x, all_spkr, marker='o', color='seagreen')
    axes[2].set_title('Hidden Layer Spike Rate per Epoch')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Spike Rate (%)')
    axes[2].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'training_curves.png'), dpi=150)
    plt.show()

    # save CSV
    log_path = os.path.join(results_dir, 'training_log.csv')
    with open(log_path, 'w', newline='') as f:
        csv.writer(f).writerows(
            [['epoch','avg_loss','train_accuracy_%','spike_rate_%','epoch_time_sec']] + log_rows
        )
    print(f'Training log saved → {log_path}')
    return log_rows

## 7. Evaluation — All Metrics

In [ ]:
def _forward_with_spikes(model, x):
    x_p = x.permute(1, 0, 2, 3, 4)
    T, B = x_p.shape[0], x_p.shape[1]
    xf = x_p.reshape(T, B, -1)
    h_states = [None] * len(model.lif_cells)
    o_state  = None
    all_hid, all_out = [], []
    with torch.no_grad():
        for t in range(T):
            z = xf[t]
            for i, cell in enumerate(model.lif_cells):
                if model.neuron_type in ('LIF', 'LIFAdEx'):
                    z = model.fc_layers[i](z); z, h_states[i] = cell(z, h_states[i])
                else:
                    z, h_states[i] = cell(z, h_states[i])
                z = model.dropouts[i](z)
            all_hid.append(z.detach().cpu())
            z = model.fc_out(z); z, o_state = model.lif_out(z, o_state)
            all_out.append(z.detach().cpu())
    hid_spk = torch.stack(all_hid, dim=0)
    out_spk = torch.stack(all_out, dim=0)
    preds   = model._decode(out_spk.to(x.device)).argmax(dim=1).cpu()
    return preds, out_spk, hid_spk

def evaluate(model, test_loader, config, log_rows, results_dir):
    model.eval()
    os.makedirs(results_dir, exist_ok=True)
    label = f"{config['neuron_type']}_{config['decoding']}_T{config['timesteps']}"
    print(f'\nEvaluating on test set...')

    all_preds, all_labels = [], []
    all_hid_spk, all_out_spk = [], []

    for images, labels in test_loader:
        images = images.to(DEVICE)
        preds, out_spk, hid_spk = _forward_with_spikes(model, images)
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
        all_hid_spk.append(hid_spk)
        all_out_spk.append(out_spk)

    all_hid_spk = torch.cat(all_hid_spk, dim=1)
    all_out_spk = torch.cat(all_out_spk, dim=1)

    # ── Metrics ───────────────────────────────────────────────
    test_acc   = round(100.0 * sum(p==l for p,l in zip(all_preds, all_labels)) / len(all_labels), 2)
    sparsity   = round(100.0 * (1.0 - all_hid_spk.mean().item()), 2)
    hid_total  = int(all_hid_spk.sum().item())
    out_total  = int(all_out_spk.sum().item())
    best_train = max(r[2] for r in log_rows)
    final_loss = log_rows[-1][1]
    total_time = round(sum(r[4] for r in log_rows), 1)
    n_params   = sum(p.numel() for p in model.parameters() if p.requires_grad)
    conv_epoch = next((r[0] for r in log_rows if r[2] >= 90.0), -1)

    # inference time
    times = []
    for i, (imgs, _) in enumerate(test_loader):
        if i >= 20: break
        imgs = imgs.to(DEVICE)
        t0 = time.perf_counter()
        with torch.no_grad(): model(imgs)
        times.append((time.perf_counter()-t0)*1000/imgs.shape[0])
    infer_time = round(sum(times)/len(times), 4)

    # confusion matrix
    cm = np.zeros((10,10), dtype=int)
    for p,l in zip(all_preds, all_labels): cm[l][p] += 1
    per_class = {i: round(100.0*cm[i][i]/cm[i].sum(), 2) if cm[i].sum()>0 else 0.0 for i in range(10)}

    # ── Print ─────────────────────────────────────────────────
    print(f'\n{"="*55}')
    print(f'  RESULTS — {label}')
    print(f'{"="*55}')
    print(f'  Test Accuracy        : {test_acc}%')
    print(f'  Best Train Accuracy  : {best_train}%')
    print(f'  Final Loss           : {final_loss}')
    print(f'  Convergence Epoch    : {conv_epoch}  (first >= 90%)')
    print(f'  Spike Sparsity       : {sparsity}%')
    print(f'  Hidden Spike Count   : {hid_total:,}')
    print(f'  Output Spike Count   : {out_total:,}')
    print(f'  Inference Time       : {infer_time} ms/sample')
    print(f'  Parameters           : {n_params:,}')
    print(f'  Total Training Time  : {total_time}s')
    print(f'{"="*55}')
    print('\n  Per-class Accuracy:')
    for d, a in per_class.items(): print(f'    Digit {d}: {a:6.2f}%  {"█"*int(a/5)}')

    # ── Confusion matrix plot ─────────────────────────────────
    fig, ax = plt.subplots(figsize=(8,7))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xlabel('Predicted digit'); ax.set_ylabel('True digit')
    ax.set_title(f'Confusion Matrix — {label}', fontsize=13)
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    for i in range(10):
        for j in range(10):
            ax.text(j, i, str(cm[i][j]), ha='center', va='center',
                    color='white' if cm[i][j] > cm.max()/2 else 'black', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'confusion_matrix.png'), dpi=150)
    plt.show()

    # ── Spike raster plot ─────────────────────────────────────
    sample_imgs, sample_lbls = next(iter(test_loader))
    s = sample_imgs[0:1].to(DEVICE)
    _, out_spk_s, hid_spk_s = _forward_with_spikes(model, s)
    hid = hid_spk_s[:,0,:].numpy()
    out = out_spk_s[:,0,:].numpy()
    activity = sample_imgs[0].sum(0).sum(0).numpy()

    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    fig.suptitle(f'Spike Raster — {label} | True: {sample_lbls[0].item()}', fontsize=13)
    axes[0].imshow(activity, cmap='hot')
    axes[0].set_title('Input Activity Map (sum over T and polarity)'); axes[0].axis('off')
    axes[1].imshow(hid[:,:64].T, aspect='auto', cmap='binary', interpolation='nearest')
    axes[1].set_xlabel('Timestep'); axes[1].set_ylabel('Hidden neuron (first 64)')
    axes[1].set_title('Hidden Layer Spike Raster')
    predicted = int(out.sum(axis=0).argmax())
    axes[2].imshow(out.T, aspect='auto', cmap='Blues', interpolation='nearest')
    axes[2].set_xlabel('Timestep'); axes[2].set_ylabel('Output neuron (digit class)')
    axes[2].set_yticks(range(10))
    axes[2].set_title(f'Output Layer — Predicted: {predicted}')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'spike_raster.png'), dpi=150)
    plt.show()

    # ── Save evaluation.md ────────────────────────────────────
    md_path = os.path.join(results_dir, 'evaluation.md')
    with open(md_path, 'w') as f:
        f.write(f'# Evaluation Results — {label}\n\n')
        f.write('## Config\n\n| Parameter | Value |\n|---|---|\n')
        for k,v in config.items():
            if k not in ('data_dir','results_dir'): f.write(f'| {k} | {v} |\n')
        f.write('\n## Training\n\n| Epoch | Loss | Train Acc % | Spike Rate % | Time (s) |\n|---|---|---|---|---|\n')
        for r in log_rows: f.write(f'| {r[0]} | {r[1]} | {r[2]} | {r[3]} | {r[4]} |\n')
        f.write('\n## Test Metrics\n\n| Metric | Value |\n|---|---|\n')
        for k,v in [
            ('Test Accuracy', f'**{test_acc}%**'),
            ('Best Train Accuracy', f'{best_train}%'),
            ('Final Train Loss', final_loss),
            ('Convergence Epoch (>=90%)', conv_epoch),
            ('Spike Sparsity', f'{sparsity}%'),
            ('Hidden Spike Count', f'{hid_total:,}'),
            ('Output Spike Count', f'{out_total:,}'),
            ('Avg Inference Time', f'{infer_time} ms/sample'),
            ('Trainable Parameters', f'{n_params:,}'),
            ('Total Training Time', f'{total_time}s'),
        ]: f.write(f'| {k} | {v} |\n')
        f.write('\n## Per-Class Accuracy\n\n| Digit | Accuracy % |\n|---|---|\n')
        for d,a in per_class.items(): f.write(f'| {d} | {a} |\n')
    print(f'\nEvaluation saved → {md_path}')

    return {'label':label,'test_acc':test_acc,'best_train':best_train,
            'final_loss':final_loss,'conv_epoch':conv_epoch,'sparsity':sparsity,
            'hid_spikes':hid_total,'out_spikes':out_total,'infer_time':infer_time,
            'n_params':n_params,'total_time':total_time,'per_class':per_class}

## 8. Experiment 1 — LIF (Baseline)

In [ ]:
cfg_lif = {**CONFIG, 'neuron_type': 'LIF'}
results_dir_lif = os.path.join(CONFIG['results_dir'], 'LIF_rate_T16_H256_L1')

model_lif  = build_model(cfg_lif).to(DEVICE)
log_lif    = train(model_lif, train_loader, cfg_lif, results_dir_lif)
res_lif    = evaluate(model_lif, test_loader, cfg_lif, log_lif, results_dir_lif)

## 9. Experiment 2 — LIFAdEx (Adaptive)

In [ ]:
cfg_adex = {**CONFIG, 'neuron_type': 'LIFAdEx'}
results_dir_adex = os.path.join(CONFIG['results_dir'], 'LIFAdEx_rate_T16_H256_L1')

model_adex = build_model(cfg_adex).to(DEVICE)
log_adex   = train(model_adex, train_loader, cfg_adex, results_dir_adex)
res_adex   = evaluate(model_adex, test_loader, cfg_adex, log_adex, results_dir_adex)

## 10. Experiment 3 — LIFRecurrent

In [ ]:
cfg_rec = {**CONFIG, 'neuron_type': 'LIFRecurrent'}
results_dir_rec = os.path.join(CONFIG['results_dir'], 'LIFRecurrent_rate_T16_H256_L1')

model_rec = build_model(cfg_rec).to(DEVICE)
log_rec   = train(model_rec, train_loader, cfg_rec, results_dir_rec)
res_rec   = evaluate(model_rec, test_loader, cfg_rec, log_rec, results_dir_rec)

## 11. Comparison — All Three Neuron Types

In [ ]:
all_results = [res_lif, res_adex, res_rec]

print(f'\n{"="*75}')
print(f'  PERFORMANCE MATRIX — Norse SNN on N-MNIST')
print(f'{"="*75}')
print(f'{"Metric":<30} {"LIF":>12} {"LIFAdEx":>12} {"LIFRecurrent":>14}')
print(f'{"-"*75}')

metrics = [
    ('Test Accuracy (%)',      'test_acc',   '{}%'),
    ('Best Train Accuracy (%)', 'best_train', '{}%'),
    ('Final Train Loss',        'final_loss', '{}'),
    ('Convergence Epoch',       'conv_epoch', 'Epoch {}'),
    ('Spike Sparsity (%)',      'sparsity',   '{}%'),
    ('Hidden Spikes (test)',    'hid_spikes', '{:,}'),
    ('Output Spikes (test)',    'out_spikes', '{:,}'),
    ('Inference Time (ms)',     'infer_time', '{} ms'),
    ('Parameters',              'n_params',   '{:,}'),
    ('Training Time (s)',       'total_time', '{}s'),
]

for name, key, fmt in metrics:
    vals = [fmt.format(r[key]) for r in all_results]
    print(f'{name:<30} {vals[0]:>12} {vals[1]:>12} {vals[2]:>14}')

print(f'{"="*75}')

# ── Bar chart comparison ──────────────────────────────────────
labels    = ['LIF', 'LIFAdEx', 'LIFRecurrent']
test_accs = [r['test_acc']  for r in all_results]
sparsities= [r['sparsity']  for r in all_results]
inf_times = [r['infer_time'] for r in all_results]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Norse SNN — N-MNIST Performance Comparison', fontsize=14)

colors = ['steelblue', 'seagreen', 'darkorange']

axes[0].bar(labels, test_accs, color=colors)
axes[0].set_title('Test Accuracy (%)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim([min(test_accs)-5, 100])
for i, v in enumerate(test_accs): axes[0].text(i, v+0.2, f'{v}%', ha='center', fontsize=10)

axes[1].bar(labels, sparsities, color=colors)
axes[1].set_title('Spike Sparsity (%)')
axes[1].set_ylabel('% Silent Neurons')
for i, v in enumerate(sparsities): axes[1].text(i, v+0.2, f'{v}%', ha='center', fontsize=10)

axes[2].bar(labels, inf_times, color=colors)
axes[2].set_title('Avg Inference Time (ms/sample)')
axes[2].set_ylabel('ms per sample')
for i, v in enumerate(inf_times): axes[2].text(i, v+0.001, f'{v}ms', ha='center', fontsize=10)

plt.tight_layout()
os.makedirs(CONFIG['results_dir'], exist_ok=True)
plt.savefig(os.path.join(CONFIG['results_dir'], 'comparison_chart.png'), dpi=150)
plt.show()
print(f'Comparison chart saved → {CONFIG["results_dir"]}/comparison_chart.png')